# Project 1 — Advanced EDA & Feature Engineering
### DecodeLabs Data Science Internship · Asadullah Ashfaq
**Goal:** Transform raw, chaotic data into a mathematically clean dataset ready for ML.

Setting Up

In [2]:
# If a library is missing in Colab, uncomment once:
# !pip install scikit-learn openpyxl -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import KNNImputer

Loading Dataset

In [3]:
from google.colab import files
uploaded = files.upload()

Saving Dataset for Data Analytics.xlsx to Dataset for Data Analytics.xlsx


Esploratory Data Analysis(EDA)

In [5]:
df = pd.read_excel('Dataset for Data Analytics.xlsx')
df.head()

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


In [8]:
print(df.shape)

(1200, 14)


In [7]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   OrderID          1200 non-null   object        
 1   Date             1200 non-null   datetime64[ns]
 2   CustomerID       1200 non-null   object        
 3   Product          1200 non-null   object        
 4   Quantity         1200 non-null   int64         
 5   UnitPrice        1200 non-null   float64       
 6   ShippingAddress  1200 non-null   object        
 7   PaymentMethod    1200 non-null   object        
 8   OrderStatus      1200 non-null   object        
 9   TrackingNumber   1200 non-null   object        
 10  ItemsInCart      1200 non-null   int64         
 11  CouponCode       891 non-null    object        
 12  ReferralSource   1200 non-null   object        
 13  TotalPrice       1200 non-null   float64       
dtypes: datetime64[ns](1), float64(2), int64(

In [14]:
print(df.describe())

                      Date     Quantity    UnitPrice  ItemsInCart  \
count                 1200  1200.000000  1200.000000  1200.000000   
mean   2024-03-22 16:58:48     2.945833   356.412750     5.485000   
min    2023-01-01 00:00:00     1.000000    11.390000     1.000000   
25%    2023-08-03 18:00:00     2.000000   186.062500     4.000000   
50%    2024-03-23 00:00:00     3.000000   364.210000     5.000000   
75%    2024-11-08 12:00:00     4.000000   521.570000     7.000000   
max    2025-06-30 00:00:00     5.000000   699.930000    10.000000   
std                    NaN     1.407557   197.177146     2.281983   

        TotalPrice    avg_value        ratio  
count  1200.000000  1200.000000  1200.000000  
mean   1053.643183   354.621692     0.016759  
min      11.390000     7.445000     0.001440  
25%     410.520000   159.706250     0.004884  
50%     823.615000   308.945000     0.008125  
75%    1578.475000   507.687500     0.015838  
max    3330.407500  1009.171875     0.330579  
st

Find Missing Values

In [20]:
print(df.isna().sum())   # missing values per column

OrderID            0
Date               0
CustomerID         0
Product            0
Quantity           0
UnitPrice          0
ShippingAddress    0
PaymentMethod      0
OrderStatus        0
TrackingNumber     0
ItemsInCart        0
CouponCode         0
ReferralSource     0
TotalPrice         0
avg_value          0
ratio              0
band               0
dtype: int64


Fill Missing Values

In [16]:
for col in df.columns:
    if df[col].isna().sum() == 0:
        continue
    if df[col].dtype == 'object':
        df[col] = df[col].fillna(df[col].mode()[0])   # text -> most common
    else:
        df[col] = df[col].fillna(df[col].median())    # number -> median

print('Missing left:', df.isna().sum().sum())

Missing left: 0


Fixing Outliers

In [17]:
for col in df.select_dtypes('number').columns:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5*iqr, q3 + 1.5*iqr
    df[col] = np.clip(df[col], low, high)   # cap extremes

print('Outliers capped')

Outliers capped


Making 3 New Features

In [22]:
df['cart_conversion'] = df['Quantity'] / df['ItemsInCart']
df['used_coupon'] = df['CouponCode'].notna().astype(int)
df['price_tier'] = pd.qcut(df['UnitPrice'], 4, labels=['Budget','Mid','Premium','Luxury'])

df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['order_month'] = df['Date'].dt.month
df['is_weekend'] = (df['Date'].dt.dayofweek >= 5).astype(int)

df.head()

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice,avg_value,ratio,band,cart_conversion,used_coupon,price_tier,order_month,is_weekend
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10,715.776460,0.008762,Top,0.714286,1,Luxury,1,0
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70,95.637619,0.013214,Low,0.666667,1,Budget,8,0
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40,691.059847,0.009080,Top,0.625000,1,Luxury,2,0
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19,115.079777,0.003660,Low,0.200000,1,Mid,10,1
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04,654.594815,0.006390,High,0.500000,1,Luxury,5,0


Saving The Data

In [23]:
df.to_csv('cleaned_dataset.csv', index=False)
files.download('cleaned_dataset.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Summary
- **Missing data:** applied the decision matrix (drop < 5%, median/mode 5–20%, KNN > 20%).
- **Outliers:** capped extreme values at the IQR fences with `np.clip()` — zero rows lost.
- **Features:** engineered 3+ new columns from existing data.
- **Output:** `cleaned_dataset.csv`, ready for the next milestone.
